# 02 · Preparação da base de conhecimento

Executa as mesmas funções usadas por `scripts/build_knowledge_base.py`, mostrando o efeito de cada regra.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 200)

In [2]:
from voc_rag.ingestion import load_raw_tables, build_review_base, clean_reviews, build_documents

tables = load_raw_tables(ROOT / "data/raw")
base = build_review_base(tables)
print(base.shape)
base.head(3)

(99224, 18)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_state,product_categories,product_categories_en,main_category,n_items,order_value,freight_value
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59,delivered,2018-01-11 15:30:49,2018-01-17 18:42:41,2018-02-02,SP,esporte_lazer,sports_leisure,esporte_lazer,2.0,370.00,27.26
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13,delivered,2018-02-28 12:25:19,2018-03-09 23:17:20,2018-03-14,SP,informatica_acessorios,computers_accessories,informatica_acessorios,1.0,79.79,8.30
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24,delivered,2018-02-03 09:56:22,2018-02-16 17:28:48,2018-03-09,BA,informatica_acessorios,computers_accessories,informatica_acessorios,1.0,149.00,45.12


In [3]:
clean_df, report = clean_reviews(base)
pd.Series(report.to_dict()).to_frame("quantidade")

,quantidade
total_reviews,99224
duplicated_review_ids_removed,814
without_text,56040
non_informative_text,419
title_only,1619
message_only,30591
title_and_message,9741
final_documents,41951


## Exemplos de normalização (título + mensagem)

In [4]:
clean_df.loc[clean_df["text_source"] == "titulo_e_mensagem",
             ["review_comment_title", "review_comment_message", "text"]].head(8)

,review_comment_title,review_comment_message,text
0,Entrega rápida!,A entrega foi super rápida e o pendente é lindo! Igual a foto mesmo!,Entrega rápida! A entrega foi super rápida e o pendente é lindo! Igual a foto mesmo!
1,SUPER RECOMENDO,"GOSTEI DA CAPINHA DE CELULAR, VEIO COMO EU ESPERAVA.","SUPER RECOMENDO. GOSTEI DA CAPINHA DE CELULAR, VEIO COMO EU ESPERAVA."
7,Bom,Meu sait preferido,Bom. Meu sait preferido
9,Entrega muito rápida,"Nota 10 pela entrega.\r\nO produto não tem a cor exatamente preta, está mais para chumbo cobreado...","Entrega muito rápida. Nota 10 pela entrega. O produto não tem a cor exatamente preta, está mais para chumbo cobreado..."
14,ENTREGA DENTRO DO PRAZO,ENTREGA DENTRO DO PRAZO..RECOMENDO!,ENTREGA DENTRO DO PRAZO..RECOMENDO!
21,Recomendo,Fiquei muito satisfeita.,Recomendo. Fiquei muito satisfeita.
25,recomendo,produto entregue no prazo e tem boa qualidade,recomendo. produto entregue no prazo e tem boa qualidade
33,super recomendado,muito bom,super recomendado. muito bom


## Textos descartados por não serem informativos

In [5]:
from voc_rag.ingestion.cleaning import normalize_text, combine_title_message, is_informative
tmp = base.drop_duplicates("review_id").copy()
tmp["text"] = [combine_title_message(normalize_text(t), normalize_text(m))
               for t, m in zip(tmp["review_comment_title"], tmp["review_comment_message"])]
tmp[(tmp["text"] != "") & ~tmp["text"].map(is_informative)]["text"].value_counts().head(20)

text
Ok       76
10       60
ok       55
.        49
...      20
OK       12
*****    12
9        10
5         9
8         9
..        6
A         4
100%      4
4         3
7         3
Ok.       3
3         3
0         3
xx        2
s         2
Name: count, dtype: int64

## Metadados derivados

In [6]:
docs = build_documents(clean_df)
print(docs["delivery_status"].value_counts(), "\n")
print(docs["sentiment"].value_counts(), "\n")
docs["main_category"].value_counts().head(15)

delivery_status
no_prazo        36286
atrasado         3809
nao_entregue     1068
cancelado         788
Name: count, dtype: int64 

sentiment
positiva    27533
negativa    10836
neutra       3582
Name: count, dtype: int64 



main_category
cama_mesa_banho           4296
beleza_saude              3498
esporte_lazer             3003
informatica_acessorios    2768
moveis_decoracao          2573
relogios_presentes        2568
utilidades_domesticas     2451
telefonia                 1873
automotivo                1737
ferramentas_jardim        1583
cool_stuff                1525
brinquedos                1455
perfumaria                1265
sem_categoria             1102
bebes                     1069
Name: count, dtype: int64

In [7]:
pd.crosstab(docs["delivery_status"], docs["review_score"], normalize="index").round(2)

review_score,1,2,3,4,5
delivery_status,,,,,
atrasado,0.64,0.09,0.09,0.06,0.12
cancelado,0.80,0.07,0.05,0.02,0.05
nao_entregue,0.75,0.08,0.06,0.04,0.06
no_prazo,0.13,0.05,0.09,0.16,0.57


**Leitura:** pedidos entregues com atraso concentram notas 1–2. Isso justifica oferecer o
filtro de **situação da entrega** como metadado, porque ele isola um recorte que o texto, sozinho,
nem sempre deixa explícito.